# *Lab 2*

## *Maira Fatima*

# *SP24-BDS-022*

# Python Lab: PDF, DOCX & API Data Processing (Tasks 1 to 9)

Install once: `pip install pdfplumber python-docx pandas nltk requests reportlab`

Set your own file paths in the config cell below. If the files don't exist, the next cell creates small sample files so everything runs out of the box.

In [5]:
import re, csv, math, os, sys, json, pprint, textwrap
from collections import Counter

# ---- CONFIG: change these to your own files ----
PDF_PATH  = "sample_document.pdf"   # multi-page PDF (5+ pages for Task 2)
DOCX_PATH = "sample_a.docx"         # docx with headings, body text and at least one table
DOCX_A    = "sample_a.docx"         # Task 6, Document A
DOCX_B    = "sample_b.docx"         # Task 6, Document B

### Optional: create sample files (skipped automatically if your files already exist)

In [6]:
def make_samples():
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas
    import docx

    # ---- sample PDF: 6 pages, page 4 intentionally blank (tests zero-word handling) ----
    para = ("Natural language processing lets computers work with human language. "
            "It powers search, translation and chatbots! Modern systems learn language patterns from data. "
            "Visit https://example.com/nlp for more info. In 2024, over 300 new models were released. "
            "Is language understanding solved? Not yet. Researchers still study language models carefully. ")
    pages = [para * n for n in (2, 3, 2, 0, 4, 3)]
    c = canvas.Canvas(PDF_PATH, pagesize=A4)
    for text in pages:
        y = 800
        for line in textwrap.wrap(text, 95):
            c.drawString(40, y, line); y -= 14
        c.showPage()
    c.save()

    # ---- sample DOCX A (has a table) and B ----
    a = docx.Document()
    a.add_heading("Introduction", 1)
    a.add_paragraph("This report explains how text data is collected and cleaned for analysis in modern projects.")
    a.add_paragraph("Data quality matters more than model size when building reliable language tools today.")
    t = a.add_table(rows=4, cols=3)
    for r, row in enumerate([["Name", "Score", "City"], ["Ali", "90", "Lahore"], ["Sara", "", "Karachi"], ["Omar", "75", ""]]):
        for k, v in enumerate(row): t.cell(r, k).text = v
    a.add_heading("Methods", 1)
    a.add_paragraph("We collected documents from public sources and removed duplicate files before processing.")
    a.add_paragraph("Data quality matters more than model size when building reliable language tools today.")
    a.add_heading("Results", 1)
    a.add_paragraph("The cleaned corpus contained fewer errors and produced clearer word frequency patterns overall.")
    a.save(DOCX_A)

    b = docx.Document()
    b.add_heading("Introduction", 1)
    b.add_paragraph("data quality matters more than model size when building reliable language tools today.  ")
    b.add_paragraph("A second team repeated the experiment using a different collection of documents.")
    b.add_heading("Results", 1)
    b.add_paragraph("Their results were similar, which suggests that the cleaning steps are reliable in practice.")
    b.add_heading("Conclusion", 1)
    b.add_paragraph("Careful preprocessing is the most important step in any language processing pipeline.")
    b.save(DOCX_B)

needed = [PDF_PATH, DOCX_PATH, DOCX_A, DOCX_B]
if not all(os.path.exists(p) for p in needed):
    make_samples()
    print("Sample files created.")
else:
    print("Using your existing files.")

Sample files created.


## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

In [7]:
import pdfplumber

# (a) page-wise extraction
with pdfplumber.open(PDF_PATH) as pdf:
    pages_text = [(page.extract_text() or "") for page in pdf.pages]
print("(a) Total pages extracted:", len(pages_text))

# (b) merge, remove punctuation + digits with re.sub, lowercase, split
full_text = " ".join(pages_text)
cleaned = re.sub(r"[^a-zA-Z\s]", " ", full_text).lower()   # removes punctuation AND digits
tokens = cleaned.split()
print("(b) Total token count:", len(tokens))

# (c) frequency dictionary without stopwords
STOPWORDS = {"the","a","an","and","or","but","is","are","was","were","be","been","of","to","in","on","at","for",
             "with","by","from","as","it","its","this","that","these","those","we","they","he","she","you","i",
             "not","so","if","than","then","into","more","most","over","their","them","our","us","can","will"}
freq = {}
for w in tokens:
    if w not in STOPWORDS:
        freq[w] = freq.get(w, 0) + 1
print("(c) Top 10 words:")
for w, n in sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"    {w:<15}{n}")

# (d) keyword search, sentence by sentence, with page numbers
keyword = input("(d) Enter a keyword to search: ").strip()
found = False
for page_no, text in enumerate(pages_text, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", text.replace("\n", " "))
    for s in sentences:
        if keyword and re.search(r"\b" + re.escape(keyword) + r"\b", s, re.IGNORECASE):
            print(f"    [Page {page_no}] {s.strip()}")
            found = True
if not found:
    print(f"    No sentence containing '{keyword}' was found.")

(a) Total pages extracted: 6
(b) Total token count: 672
(c) Top 10 words:
    language       70
    models         28
    natural        14
    processing     14
    lets           14
    computers      14
    work           14
    human          14
    powers         14
    search         14
    No sentence containing '' was found.


## Task 2: Structured Page Metadata Extraction and CSV Export

In [8]:
# (a) build pages_data
pages_data = []
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        raw = page.extract_text() or ""
        pages_data.append({
            "page_number": i,
            "raw_text": raw,
            "word_count": len(raw.split()),
            "char_count": len(re.sub(r"\s", "", raw)),   # characters excluding spaces/whitespace
        })
print("(a) First 2 entries:")
for d in pages_data[:2]:
    print("   ", {**d, "raw_text": d["raw_text"][:80] + "..."})   # raw_text shortened only for display

# (b) highest, lowest non-zero, average
if len(pages_data) == 0:
    print("(b) No pages found.")
else:
    highest = max(pages_data, key=lambda d: d["word_count"])
    print(f"\n(b)(i) Highest word count: page {highest['page_number']} ({highest['word_count']} words)")
    print(highest["raw_text"])

    non_zero = [d for d in pages_data if d["word_count"] > 0]
    if non_zero:
        lowest = min(non_zero, key=lambda d: d["word_count"])
        print(f"\n(b)(ii) Lowest non-zero word count: page {lowest['page_number']} ({lowest['word_count']} words)")
    else:
        print("\n(b)(ii) Every page has zero words.")
    zero_pages = [d["page_number"] for d in pages_data if d["word_count"] == 0]
    if zero_pages:
        print("        Pages with zero words (skipped for the lowest count):", zero_pages)

    avg = sum(d["word_count"] for d in pages_data) / len(pages_data)
    print(f"(b)(iii) Average word count across all pages: {avg:.2f}")

# (c) case-insensitive phrase search with custom exception
class PhraseNotFoundError(Exception):
    pass

phrase = input("\n(c) Enter a search phrase: ").strip()
try:
    hits = []
    for d in pages_data:
        n = len(re.findall(re.escape(phrase), d["raw_text"], flags=re.IGNORECASE)) if phrase else 0
        if n:
            hits.append((d["page_number"], n))
    if not hits:
        raise PhraseNotFoundError(f"The phrase '{phrase}' was not found in any page.")
    for page_no, n in hits:
        print(f"    Page {page_no}: {n} occurrence(s)")
except PhraseNotFoundError as e:
    print("    PhraseNotFoundError caught:", e)

# (d) write and re-read CSV with the csv module
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for d in pages_data:
        writer.writerow([d["page_number"], d["word_count"], d["char_count"]])

print("\n(d) Contents of pdf_summary.csv:")
with open("pdf_summary.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print("   ", row)

(a) First 2 entries:
    {'page_number': 1, 'raw_text': 'Natural language processing lets computers work with human language. It powers s...', 'word_count': 94, 'char_count': 598}
    {'page_number': 2, 'raw_text': 'Natural language processing lets computers work with human language. It powers s...', 'word_count': 141, 'char_count': 897}

(b)(i) Highest word count: page 5 (188 words)
Natural language processing lets computers work with human language. It powers search,
translation and chatbots! Modern systems learn language patterns from data. Visit
https://example.com/nlp for more info. In 2024, over 300 new models were released. Is language
understanding solved? Not yet. Researchers still study language models carefully. Natural
language processing lets computers work with human language. It powers search, translation and
chatbots! Modern systems learn language patterns from data. Visit https://example.com/nlp for
more info. In 2024, over 300 new models were released. Is language und

## Task 3: PDF Text Preprocessing Pipeline for NLP

In [9]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)

# (a) extract all text and clean it in the required order
with pdfplumber.open(PDF_PATH) as pdf:
    raw_text = "\n".join((p.extract_text() or "") for p in pdf.pages)

text = raw_text
before_chars = len(text)
text = re.sub(r"https?://\S+|www\.\S+", " ", text)     # (i) URLs
text = re.sub(r"\d+", "", text)                          # (ii) digits
text = re.sub(r"[^A-Za-z. ]", " ", text)                 # (iii) keep letters, spaces, full stops (newlines become spaces)
text = re.sub(r" {2,}", " ", text)                       # (iv) collapse spaces
print("(a) Characters before cleaning:", before_chars)
print("    Characters after cleaning :", len(text))

# (b) lowercase, split into sentences, strip, drop empties
text = text.lower()
sentences = re.split(r"[.!?]", text)
print("\n(b) Sentence count before removing empties:", len(sentences))
sentences = [s.strip() for s in sentences]
sentences = [s for s in sentences if s]
print("    Sentence count after removing empties :", len(sentences))

# (c) tokenize, remove stopwords, keep sentences with >= 5 tokens left
stop_en = set(stopwords.words("english"))
print("\n(c) Sentence count before filter:", len(sentences))
kept = []
for s in sentences:
    toks = [w for w in re.findall(r"[a-z]+", s) if w not in stop_en]
    if len(toks) >= 5:
        kept.append(s)            # the sentence itself is kept as written; tokens are only used for the length check
print("    Sentence count after filter :", len(kept))
print("    3 sample sentences:")
for s in kept[:3]:
    print("     -", s)

# (d) write, re-read, report
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for s in kept:
        f.write(s + "\n")

with open("nlp_ready_output.txt", encoding="utf-8") as f:
    lines = [ln.rstrip("\n") for ln in f]
print("\n(d)(i) Total lines written:", len(lines))
if lines:
    print("    (ii) Longest sentence :", max(lines, key=len))
    print("    (iii) Shortest sentence:", min(lines, key=len))
else:
    print("    No sentences survived the filter.")

(a) Characters before cleaning: 4844
    Characters after cleaning : 4325

(b) Sentence count before removing empties: 85
    Sentence count after removing empties : 84

(c) Sentence count before filter: 84
    Sentence count after filter : 42
    3 sample sentences:
     - natural language processing lets computers work with human language
     - it powers search translation and chatbots modern systems learn language patterns from data
     - researchers still study language models carefully

(d)(i) Total lines written: 42
    (ii) Longest sentence : it powers search translation and chatbots modern systems learn language patterns from data
    (iii) Shortest sentence: researchers still study language models carefully


## Task 4: DOCX Paragraph Classification and Statistical Analysis

In [10]:
import docx

doc = docx.Document(DOCX_PATH)

# (a) text + style per paragraph
paras = [{"text": p.text, "style": p.style.name} for p in doc.paragraphs]
print("(a) Total paragraphs:", len(paras))
print("    Unique style names:", sorted({p["style"] for p in paras}))

# (b) classify into exactly three groups
headings, body, other = [], [], []
for p in paras:
    if not p["text"].strip():
        other.append(p)                                   # blank paragraphs always go to Other/Empty
    elif p["style"].startswith("Heading"):
        headings.append(p)
    elif p["style"] == "Normal" or "Body" in p["style"]:
        body.append(p)
    else:
        other.append(p)
print(f"\n(b) Headings: {len(headings)} | Body: {len(body)} | Other/Empty: {len(other)}")

# (c) stats for body paragraphs with more than 5 words
qualifying = []
for p in body:
    wc = len(p["text"].split())
    if wc > 5:
        qualifying.append({**p, "words": wc, "chars": len(re.sub(r"\s", "", p["text"]))})
if qualifying:
    hi = max(qualifying, key=lambda q: q["words"])
    lo = min(qualifying, key=lambda q: q["words"])
    mean_wc = sum(q["words"] for q in qualifying) / len(qualifying)
    print(f"\n(c)(i) Highest word count ({hi['words']} words, {hi['chars']} chars):\n     {hi['text']}")
    print(f"    (ii) Lowest word count above 5 ({lo['words']} words, {lo['chars']} chars):\n     {lo['text']}")
    print(f"    (iii) Mean word count: {mean_wc:.2f}")
else:
    print("\n(c) No body paragraphs with more than 5 words.")

# (d) nested dictionary by style, sorted by count (desc), max 2 texts shown
by_style = {}
for p in paras:
    entry = by_style.setdefault(p["style"], {"count": 0, "texts": []})
    entry["count"] += 1
    entry["texts"].append(p["text"])
by_style = dict(sorted(by_style.items(), key=lambda kv: kv[1]["count"], reverse=True))
for entry in by_style.values():
    if entry["count"] > 2:
        entry["texts"] = entry["texts"][:2]      # count stays the true total
print("\n(d) Style summary:")
pprint.pprint(by_style, sort_dicts=False, width=100)

(a) Total paragraphs: 8
    Unique style names: ['Heading 1', 'Normal']

(b) Headings: 3 | Body: 5 | Other/Empty: 0

(c)(i) Highest word count (15 words, 78 chars):
     This report explains how text data is collected and cleaned for analysis in modern projects.
    (ii) Lowest word count above 5 (12 words, 78 chars):
     We collected documents from public sources and removed duplicate files before processing.
    (iii) Mean word count: 13.20

(d) Style summary:
{'Normal': {'count': 5,
            'texts': ['This report explains how text data is collected and cleaned for analysis in '
                      'modern projects.',
                      'Data quality matters more than model size when building reliable language '
                      'tools today.']},
 'Heading 1': {'count': 3, 'texts': ['Introduction', 'Methods']}}


## Task 5: DOCX Table Extraction and Pandas Integration

In [11]:
import pandas as pd
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph

doc = docx.Document(DOCX_PATH)

# (a) every table as a list of lists
all_tables = []
for t in doc.tables:
    all_tables.append([[cell.text.strip() for cell in row.cells] for row in t.rows])
print("(a) Tables found:", len(all_tables))
for i, t in enumerate(all_tables, start=1):
    cols = len(t[0]) if t else 0
    print(f"    Table {i}: {len(t)} rows x {cols} columns")

# (b) first table -> DataFrame, count empty cells per column
if all_tables and len(all_tables[0]) > 1:
    first = all_tables[0]
    df = pd.DataFrame(first[1:], columns=first[0])
    print("\n(b) DataFrame:")
    print(df)
    is_empty = df.apply(lambda col: col.map(lambda x: x is None or str(x).strip() == ""))
    print("\n    Empty/None cells per column:")
    print(is_empty.sum())
else:
    print("\n(b) The first table has no data rows.")

# (c) paragraphs that appear before the first table
pre_table = []
for child in doc.element.body.iterchildren():
    if child.tag == qn("w:tbl"):
        break
    if child.tag == qn("w:p"):
        pre_table.append(Paragraph(child, doc).text)
print("\n(c) Paragraphs before first table:", len(pre_table))

# (d) write both sections to docx_extracted.txt
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_table:
        f.write(p + "\n")
    f.write("\n--- TABLE DATA ---\n")
    if all_tables:
        for row in all_tables[0]:
            f.write(" | ".join(row) + "\n")
print("\n(d) docx_extracted.txt written:")
print(open("docx_extracted.txt", encoding="utf-8").read())

(a) Tables found: 1
    Table 1: 4 rows x 3 columns

(b) DataFrame:
   Name Score     City
0   Ali    90   Lahore
1  Sara        Karachi
2  Omar    75         

    Empty/None cells per column:
Name     0
Score    1
City     1
dtype: int64

(c) Paragraphs before first table: 3

(d) docx_extracted.txt written:
--- PRE-TABLE PARAGRAPHS ---
Introduction
This report explains how text data is collected and cleaned for analysis in modern projects.
Data quality matters more than model size when building reliable language tools today.

--- TABLE DATA ---
Name | Score | City
Ali | 90 | Lahore
Sara |  | Karachi
Omar | 75 | 



## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

In [12]:
docA, docB = docx.Document(DOCX_A), docx.Document(DOCX_B)

# (a) headings + set operations
headA = [p.text.strip() for p in docA.paragraphs if p.style.name.startswith("Heading") and p.text.strip()]
headB = [p.text.strip() for p in docB.paragraphs if p.style.name.startswith("Heading") and p.text.strip()]
setA, setB = set(headA), set(headB)
print("(a)(i)   Common headings      :", sorted(setA & setB))
print("    (ii)  Only in Document A   :", sorted(setA - setB))
print("    (iii) Only in Document B   :", sorted(setB - setA))

# (b) Normal paragraphs, merge, dedupe (stripped + lowercased comparison). Blank paragraphs are ignored.
normA = [p.text for p in docA.paragraphs if p.style.name == "Normal" and p.text.strip()]
normB = [p.text for p in docB.paragraphs if p.style.name == "Normal" and p.text.strip()]
merged = normA + normB
seen, unique_paras = set(), []
for p in merged:
    key = p.strip().lower()
    if key not in seen:
        seen.add(key)
        unique_paras.append(p.strip())
print(f"\n(b) Paragraphs before dedup: {len(merged)} | after dedup: {len(unique_paras)}")

# (c) tokenize, combined Counter, remove stopwords
STOP6 = {"the","a","an","and","or","of","to","in","on","for","with","is","are","was","were","be","that","this",
         "it","as","by","at","from","than","more","which","their","any","we","before","using"}
all_tokens = []
for p in unique_paras:
    all_tokens += re.sub(r"[^a-zA-Z\s]", " ", p).lower().split()
word_freq = Counter(w for w in all_tokens if w not in STOP6)
print("\n(c) Top 20 words:")
for w, n in word_freq.most_common(20):
    print(f"    {w:<15}{n}")

# (d) summary
para_lengths = [len(p.split()) for p in unique_paras]
avg_len = sum(para_lengths) / len(para_lengths) if para_lengths else 0
longest = max(unique_paras, key=lambda p: len(p.split())) if unique_paras else "N/A"
print("\n(d) Summary")
print("    (i)   Total headings (both docs)       :", len(headA) + len(headB))
print("    (ii)  Unique body paragraphs           :", len(unique_paras))
print("    (iii) Vocabulary size (unique words)   :", len(set(all_tokens)))
print(f"    (iv)  Average paragraph length (words) : {avg_len:.2f}")
print("    (v)   Longest paragraph:", longest)

(a)(i)   Common headings      : ['Introduction', 'Results']
    (ii)  Only in Document A   : ['Methods']
    (iii) Only in Document B   : ['Conclusion']

(b) Paragraphs before dedup: 9 | after dedup: 7

(c) Top 20 words:
    data           2
    collected      2
    cleaned        2
    reliable       2
    language       2
    documents      2
    processing     2
    report         1
    explains       1
    how            1
    text           1
    analysis       1
    modern         1
    projects       1
    quality        1
    matters        1
    model          1
    size           1
    when           1
    building       1

(d) Summary
    (i)   Total headings (both docs)       : 6
    (ii)  Unique body paragraphs           : 7
    (iii) Vocabulary size (unique words)   : 75
    (iv)  Average paragraph length (words) : 13.00
    (v)   Longest paragraph: This report explains how text data is collected and cleaned for analysis in modern projects.


## Task 7: Multi-Endpoint API Data Joining and Analysis

Tasks 7 to 9 need internet access to jsonplaceholder.typicode.com.

In [13]:
import requests

def fetch_json(url):
    r = requests.get(url, timeout=15)
    if r.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {r.status_code}")
    return r.json()

# (a)
users = fetch_json("https://jsonplaceholder.typicode.com/users")
posts = fetch_json("https://jsonplaceholder.typicode.com/posts")
print("(a) Users:", len(users), "| Posts:", len(posts))

# (b) join users.id with posts.userId
joined = {}
for u in users:
    titles = [p["title"] for p in posts if p["userId"] == u["id"]]
    joined[u["name"]] = {"email": u["email"], "post_count": len(titles), "titles": titles}
print("\n(b) First 3 users:")
for name in list(joined)[:3]:
    print(name)
    pprint.pprint(joined[name], sort_dicts=False, width=110)

# (c) user with most posts
top_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user = next(u for u in users if u["name"] == top_name)
print("\n(c)(i)   Name   :", top_name)
print("    (ii)  Email  :", top_user["email"])
print("    (iii) Company:", top_user["company"]["name"])
print("    (iv)  Post titles:")
for i, t in enumerate(joined[top_name]["titles"], start=1):
    print(f"        {i}. {t}")

# (d) filters: bs contains 'synergies' OR city longer than 8 characters
print("\n(d) Matching users:")
for u in users:
    if "synergies" in u["company"]["bs"].lower() or len(u["address"]["city"]) > 8:
        print(f"    {u['name']} | {u['address']['city']} | posts: {joined[u['name']]['post_count']}")

(a) Users: 10 | Posts: 100

(b) First 3 users:
Leanne Graham
{'email': 'Sincere@april.biz',
 'post_count': 10,
 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
            'qui est esse',
            'ea molestias quasi exercitationem repellat qui ipsa sit aut',
            'eum et est occaecati',
            'nesciunt quas odio',
            'dolorem eum magni eos aperiam quia',
            'magnam facilis autem',
            'dolorem dolore est ipsam',
            'nesciunt iure omnis dolorem tempora et accusantium',
            'optio molestias id quia eum']}
Ervin Howell
{'email': 'Shanna@melissa.tv',
 'post_count': 10,
 'titles': ['et ea vero quia laudantium autem',
            'in quibusdam tempore odit est dolorem',
            'dolorum ut in voluptas mollitia et saepe quo animi',
            'voluptatem eligendi optio',
            'eveniet quod temporibus',
            'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio

## Task 8: To-Do Completion Rate Analysis Across Users

In [14]:
def fail(msg):
    print("ERROR:", msg)
    raise SystemExit(msg)

# (a) fetch + validate
resp = requests.get("https://jsonplaceholder.typicode.com/todos", timeout=15)
if resp.status_code != 200:
    fail(f"Expected status 200 but got {resp.status_code}")
todos = resp.json()
if not isinstance(todos, list):
    fail("Parsed response is not a list")
if len(todos) < 1:
    fail("The list has no records")
print("(a) Total todo records:", len(todos))

# (b) group by userId
stats = {}
for t in todos:
    s = stats.setdefault(t["userId"], {"total": 0, "completed": 0, "incomplete": 0})
    s["total"] += 1
    s["completed" if t["completed"] else "incomplete"] += 1
print("\n(b) Per-user stats:")
for uid in sorted(stats):
    s = stats[uid]
    print(f"    User {uid}: {s} | completed + incomplete == total: {s['completed'] + s['incomplete'] == s['total']}")

# (c) completion rates
rates = {uid: round(s["completed"] / s["total"] * 100, 1) for uid, s in stats.items()}
best = max(rates, key=rates.get)
worst = min(rates, key=rates.get)
print(f"\n(c)(i)   Highest rate: user {best} ({rates[best]}%)")
print(f"    (ii)  Lowest rate : user {worst} ({rates[worst]}%)")
print(f"    (iii) Overall average rate: {sum(rates.values()) / len(rates):.1f}%")

# (d) names + formatted table
users8 = fetch_json("https://jsonplaceholder.typicode.com/users")
lookup = {u["id"]: u["name"] for u in users8}
ranked = sorted(stats, key=lambda uid: rates[uid], reverse=True)
print("\n(d)")
header = f"{'Rank':<6}{'Name':<26}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %':<18}"
print(header)
print("-" * len(header))
for rank, uid in enumerate(ranked, start=1):
    s = stats[uid]
    print(f"{rank:<6}{lookup.get(uid, 'Unknown'):<26}{s['total']:<14}{s['completed']:<12}{rates[uid]:<18}")

(a) Total todo records: 200

(b) Per-user stats:
    User 1: {'total': 20, 'completed': 11, 'incomplete': 9} | completed + incomplete == total: True
    User 2: {'total': 20, 'completed': 8, 'incomplete': 12} | completed + incomplete == total: True
    User 3: {'total': 20, 'completed': 7, 'incomplete': 13} | completed + incomplete == total: True
    User 4: {'total': 20, 'completed': 6, 'incomplete': 14} | completed + incomplete == total: True
    User 5: {'total': 20, 'completed': 12, 'incomplete': 8} | completed + incomplete == total: True
    User 6: {'total': 20, 'completed': 6, 'incomplete': 14} | completed + incomplete == total: True
    User 7: {'total': 20, 'completed': 9, 'incomplete': 11} | completed + incomplete == total: True
    User 8: {'total': 20, 'completed': 11, 'incomplete': 9} | completed + incomplete == total: True
    User 9: {'total': 20, 'completed': 8, 'incomplete': 12} | completed + incomplete == total: True
    User 10: {'total': 20, 'completed': 12, 'incomp

## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [15]:
# (a) flatten nested user
def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

users9 = fetch_json("https://jsonplaceholder.typicode.com/users")
flat = [flatten_user(u) for u in users9]
print("(a) Flattened users:")
pprint.pprint(flat, sort_dicts=False, width=120)

# (b) DataFrame
df = pd.DataFrame(flat)
df["lat"] = df["lat"].astype(float)
df["lng"] = df["lng"].astype(float)
df["distance_from_origin"] = df.apply(lambda r: math.sqrt(r["lat"] ** 2 + r["lng"] ** 2), axis=1)
with pd.option_context("display.max_columns", None, "display.width", 250):
    print("\n(b) DataFrame:")
    print(df)
far = df.loc[df["distance_from_origin"].idxmax()]
print(f"\n    Greatest distance from origin: {far['name']} ({far['distance_from_origin']:.2f})")

# (c) hemispheres
north = df[df["lat"] > 0]
south = df[df["lat"] < 0]
print("\n(c) Northern Hemisphere:", list(north["name"]))
print("    Southern Hemisphere:", list(south["name"]))
n_avg = north["distance_from_origin"].mean() if len(north) else float("nan")
s_avg = south["distance_from_origin"].mean() if len(south) else float("nan")
print(f"    Average distance North: {n_avg:.2f} | South: {s_avg:.2f}")
if len(north) and len(south):
    print("    Higher average distance:", "Northern" if n_avg > s_avg else "Southern", "Hemisphere")

# (d) comments per post
posts9 = fetch_json("https://jsonplaceholder.typicode.com/posts")
comments = fetch_json("https://jsonplaceholder.typicode.com/comments")
comment_counts = Counter(c["postId"] for c in comments)
result = [(p["id"], p["title"][:40], comment_counts.get(p["id"], 0)) for p in posts9]
result.sort(key=lambda t: (-t[2], t[0]))     # most comments first, ties broken by post id
print("\n(d) Top 5 most-commented posts:")
for t in result[:5]:
    print("   ", t)
print("    Bottom 5 least-commented posts:")
for t in result[-5:]:
    print("   ", t)

(a) Flattened users:
[{'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'city': 'Gwenborough',
  'zipcode': '92998-3874',
  'lat': '-37.3159',
  'lng': '81.1496',
  'company_name': 'Romaguera-Crona',
  'company_catchphrase': 'Multi-layered client-server neural-net'},
 {'id': 2,
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'email': 'Shanna@melissa.tv',
  'city': 'Wisokyburgh',
  'zipcode': '90566-7771',
  'lat': '-43.9509',
  'lng': '-34.4618',
  'company_name': 'Deckow-Crist',
  'company_catchphrase': 'Proactive didactic contingency'},
 {'id': 3,
  'name': 'Clementine Bauch',
  'username': 'Samantha',
  'email': 'Nathan@yesenia.net',
  'city': 'McKenziehaven',
  'zipcode': '59590-4157',
  'lat': '-68.6102',
  'lng': '-47.0653',
  'company_name': 'Romaguera-Jacobson',
  'company_catchphrase': 'Face to face bifurcated interface'},
 {'id': 4,
  'name': 'Patricia Lebsack',
  'username': 'Karianne',
  'email': 'Julianne.OConner@kory.org',